# Semana 6 — Tokenization, Embeddings e um Bigram Language Model
**Fase 2: Deep Learning e o Ecossistema Hugging Face** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/02-Phase-2-Deep-Learning/notebooks/semana_06_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-06-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/02-Phase-2-Deep-Learning/docs/Semana-06-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_06_gabarito.ipynb`.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(1337); print(device)

## Passo 1 — O corpus

In [ ]:
# --- Corpus: tenta baixar o Tiny Shakespeare (1 MB); sem internet, usa um texto curto embutido ---
import urllib.request
URL_CORPUS = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    texto = urllib.request.urlopen(URL_CORPUS, timeout=15).read().decode("utf-8")
except Exception as e:
    print("Download indisponível, usando texto embutido:", type(e).__name__)
    texto = ("""Um modelo de linguagem aprende a prever o próximo símbolo a partir dos símbolos anteriores.
Quando o texto é pequeno, o modelo decora; quando o texto é grande, o modelo generaliza.
A arquiteta de dados perguntou qual era a métrica correta, e o engenheiro respondeu que dependia do custo de cada erro.
Quem quer entender um sistema precisa construir uma versão pequena dele, quebrar essa versão e consertar de novo.
O vetor que representa uma frase fica perto do vetor de outra frase quando as duas querem dizer quase a mesma coisa.
A cada passo o gradiente aponta a direção em que o erro cresce, e o otimizador caminha para o lado contrário.
Que o modelo erre é esperado; que ninguém meça o erro é que é o problema de arquitetura.
""" * 40)
# Para treinar em português, troque `texto` pelo conteúdo de qualquer arquivo .txt (ex.: um livro em domínio público).
print(len(texto), "caracteres\n", texto[:300])

## Passo 2 — Tokenizer de caracteres
Construa o vocabulário (caracteres únicos, ordenados), os dicionários `stoi`/`itos` e as funções `encode`/`decode`.

In [ ]:
# [SEU CÓDIGO AQUI] defina chars, V, stoi, itos, encode, decode
assert decode(encode(texto[:50])) == texto[:50]
print("vocabulário:", V, "| perda de um chute uniforme: ln(V) =", round(math.log(V), 3))

## Passo 3 — Train/validation split e batches
`get_batch` sorteia `B` posições e devolve `x` (janelas de `T` tokens) e `y` (as mesmas janelas deslocadas 1 posição).

In [ ]:
dados = torch.tensor(encode(texto), dtype=torch.long)
n = int(0.9 * len(dados)); treino, val = dados[:n], dados[n:]

def get_batch(split, B=64, T=32):
    d = treino if split == "treino" else val
    ix = torch.randint(len(d) - T - 1, (B,))
    # [SEU CÓDIGO AQUI] monte x e y com torch.stack
    return x.to(device), y.to(device)

xb, yb = get_batch("treino"); print(xb.shape, yb.shape)
print("entrada:", repr(decode(xb[0].tolist())), "\nalvo   :", repr(decode(yb[0].tolist())))

## Passo 4 — O modelo bigrama
Uma única tabela `V × V`: a linha do token atual já contém os logits do próximo token.

In [ ]:
class Bigrama(nn.Module):
    def __init__(self, V):
        super().__init__()
        # [SEU CÓDIGO AQUI] self.tabela = nn.Embedding(...)
    def forward(self, idx, alvos=None):
        logits = self.tabela(idx)                       # (B, T, V)
        if alvos is None: return logits, None
        # [SEU CÓDIGO AQUI] achate logits para (B*T, V) e alvos para (B*T) e use F.cross_entropy
        return logits, loss

    @torch.no_grad()
    def gerar(self, idx, n_tokens, temperatura=1.0):
        for _ in range(n_tokens):
            logits, _ = self(idx[:, -1:])
            # [SEU CÓDIGO AQUI] softmax com temperature e sampling com torch.multinomial
            idx = torch.cat([idx, prox], dim=1)
        return idx

m = Bigrama(V).to(device)
_, loss = m(xb, yb); print("perda inicial:", round(loss.item(), 3), "(esperado ≈", round(math.log(V), 3), ")")
inicio = torch.zeros((1, 1), dtype=torch.long, device=device)
print("antes do treino:", repr(decode(m.gerar(inicio, 80)[0].tolist())))

## Passo 5 — Treino

In [ ]:
@torch.no_grad()
def perda_media(modelo, split, n=50):
    modelo.eval(); l = sum(modelo(*get_batch(split))[1].item() for _ in range(n)) / n; modelo.train(); return l

def treinar(modelo, passos=3000, lr=1e-2):
    opt = torch.optim.AdamW(modelo.parameters(), lr=lr); hist = []
    for p in range(passos):
        # [SEU CÓDIGO AQUI] um passo de treino
        if p % 500 == 0 or p == passos - 1:
            hist.append((p, perda_media(modelo, "treino"), perda_media(modelo, "val"))); print(hist[-1])
    return hist

h = treinar(m)
print("perplexidade (val):", round(math.exp(h[-1][2]), 2))

## Passo 6 — Generation e temperature

In [ ]:
for T in (0.5, 1.0, 1.5):
    print(f"\n--- temperatura {T} ---\n", decode(m.gerar(inicio, 200, temperatura=T)[0].tolist()))

## Passo 7 — O que a tabela aprendeu
Cada linha é `P(próximo | atual)`. Veja o que costuma vir depois de `q`.

In [ ]:
with torch.no_grad():
    P = F.softmax(m.tabela.weight, dim=1).cpu()
c = "q" if "q" in stoi else chars[-1]
top = torch.topk(P[stoi[c]], 5)
print(f"depois de {c!r}:", [(itos[i.item()], round(p.item(), 2)) for p, i in zip(top.values, top.indices)])
plt.figure(figsize=(6, 6)); plt.imshow(P, cmap="Blues"); plt.title("P(próximo | atual)"); plt.show()

## Passo 8 — Embeddings densos
Troque a tabela `V × V` por um embedding de dimensão 16 seguido de uma layer linear. Mesma tarefa, mas agora cada caractere tem um vetor de 16 dimensões que podemos inspecionar.

In [ ]:
class BigramaDenso(Bigrama):
    def __init__(self, V, d=16):
        nn.Module.__init__(self)
        # [SEU CÓDIGO AQUI] self.emb e self.saida
    def forward(self, idx, alvos=None):
        logits = self.saida(self.emb(idx))
        loss = None if alvos is None else F.cross_entropy(logits.view(-1, logits.size(-1)), alvos.view(-1))
        return logits, loss

md = BigramaDenso(V).to(device); _ = treinar(md)
E = F.normalize(md.emb.weight.detach().cpu(), dim=1)
for c in [x for x in "ae.r" if x in stoi]:
    sims = E @ E[stoi[c]]; viz = torch.topk(sims, 5).indices[1:]
    print(f"vizinhos de {c!r}:", [itos[i.item()] for i in viz])

**Reflexão:** vogais tendem a ficar próximas entre si. Ninguém ensinou isso ao modelo: emergiu da tarefa de prever o próximo caractere. É o mesmo mecanismo que, em escala, dá significado aos embeddings de um LLM.